In [2]:
# ==========================================================
# Module 2.14 - Dataset Intake & Source Validation
# ==========================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("DATASET INTAKE & SOURCE VALIDATION")
print("=" * 70)

# ==========================================================
# Configuration
# ==========================================================

DATA_FOLDER = Path("../data")
OUTPUT_FOLDER = Path("../output")

OUTPUT_FOLDER.mkdir(exist_ok=True)

FILE_NAME = "Combined_Data.xlsx"
FILE_PATH = DATA_FOLDER / FILE_NAME

EXPECTED_SHEETS = [
    "Support Tickets",
    "Jira Issues",
    "Survey Responses"
]

# ==========================================================
# File Validation
# ==========================================================

print("\nChecking Dataset...")

if not FILE_PATH.exists():
    raise FileNotFoundError(f"\nDataset not found:\n{FILE_PATH.resolve()}")

print("Dataset Found")

file_size = round(FILE_PATH.stat().st_size / (1024 * 1024), 2)

print(f"File Size : {file_size} MB")
print(f"Extension : {FILE_PATH.suffix}")

# ==========================================================
# Load Workbook
# ==========================================================

print("\nLoading Workbook...")

workbook = pd.read_excel(FILE_PATH, sheet_name=None)

print("Workbook Loaded Successfully")

available_sheets = list(workbook.keys())

print("\nAvailable Sheets")
for sheet in available_sheets:
    print(f" • {sheet}")

# ==========================================================
# Workbook Summary
# ==========================================================

summary = []

for sheet, df in workbook.items():

    summary.append({

        "Sheet Name": sheet,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Memory(MB)": round(
            df.memory_usage(deep=True).sum()/1024**2,
            2
        )

    })

summary_df = pd.DataFrame(summary)

print("\nWorkbook Summary")
display(summary_df)

# ==========================================================
# Expected Sheet Validation
# ==========================================================

validation = []

for sheet in EXPECTED_SHEETS:

    validation.append({

        "Sheet": sheet,
        "Exists": sheet in available_sheets

    })

sheet_validation = pd.DataFrame(validation)

print("\nSheet Validation")
display(sheet_validation)

# ==========================================================
# Schema Validation
# ==========================================================

schema = []

for sheet, df in workbook.items():

    for col in df.columns:

        schema.append({

            "Sheet": sheet,
            "Column": col,
            "Data Type": str(df[col].dtype),
            "Null Count": int(df[col].isna().sum()),
            "Unique Values": int(df[col].nunique())

        })

schema_df = pd.DataFrame(schema)

print("\nSchema Sample")
display(schema_df.head(20))

# ==========================================================
# Duplicate Sheet Check
# ==========================================================

duplicate_sheet_names = len(available_sheets) != len(set(available_sheets))

print("\nDuplicate Sheet Names :", duplicate_sheet_names)

# ==========================================================
# Mandatory Column Validation
# ==========================================================

mandatory = {
    "Support Tickets": ["Ticket Id"],
    "Jira Issues": ["Key"],
    "Survey Responses": ["Response ID"]
}

mandatory_report = []

for sheet, cols in mandatory.items():

    if sheet not in workbook:
        continue

    df = workbook[sheet]

    for col in cols:

        mandatory_report.append({

            "Sheet": sheet,
            "Column": col,
            "Exists": col in df.columns

        })

mandatory_df = pd.DataFrame(mandatory_report)

print("\nMandatory Column Validation")
display(mandatory_df)

# ==========================================================
# Workbook Statistics
# ==========================================================

stats = []

for sheet, df in workbook.items():

    stats.append({

        "Sheet": sheet,

        "Rows": len(df),

        "Columns": len(df.columns),

        "Missing Cells": int(df.isna().sum().sum()),

        "Duplicate Rows": int(df.duplicated().sum()),

        "Numeric Columns":
            len(df.select_dtypes(include=np.number).columns),

        "Categorical Columns":
            len(df.select_dtypes(include=["object", "string"]).columns),

        "Date Columns":
            len(df.select_dtypes(include=["datetime64[ns]"]).columns)

    })

stats_df = pd.DataFrame(stats)

print("\nWorkbook Statistics")
display(stats_df)

# ==========================================================
# Overall Validation Status
# ==========================================================

overall_status = []

for _, row in sheet_validation.iterrows():
    if not row["Exists"]:
        overall_status.append(f"Missing Sheet: {row['Sheet']}")

for _, row in mandatory_df.iterrows():
    if not row["Exists"]:
        overall_status.append(
            f"Missing Column: {row['Column']} ({row['Sheet']})"
        )

if len(overall_status) == 0:
    overall_result = "PASS"
else:
    overall_result = "FAIL"

print("\nOverall Validation Result :", overall_result)

if overall_status:
    print("\nIssues Found")
    for issue in overall_status:
        print("-", issue)
else:
    print("No validation issues detected.")

# ==========================================================
# Save Reports
# ==========================================================

print("\nSaving Reports...")

with pd.ExcelWriter(
    OUTPUT_FOLDER / "Validation_Report.xlsx",
    engine="openpyxl"
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Workbook Summary",
        index=False
    )

    sheet_validation.to_excel(
        writer,
        sheet_name="Sheet Validation",
        index=False
    )

    schema_df.to_excel(
        writer,
        sheet_name="Schema Validation",
        index=False
    )

    mandatory_df.to_excel(
        writer,
        sheet_name="Mandatory Columns",
        index=False
    )

    stats_df.to_excel(
        writer,
        sheet_name="Statistics",
        index=False
    )

summary_df.to_excel(
    OUTPUT_FOLDER / "Workbook_Summary.xlsx",
    index=False
)

print("\nReports Generated Successfully!")

print("\nGenerated Files")

print("output/")
print(" ├── Validation_Report.xlsx")
print(" └── Workbook_Summary.xlsx")

print("\nNotebook 01 Completed Successfully")
print("=" * 70)

DATASET INTAKE & SOURCE VALIDATION

Checking Dataset...
Dataset Found
File Size : 1.05 MB
Extension : .xlsx

Loading Workbook...
Workbook Loaded Successfully

Available Sheets
 • Support Tickets
 • Jira Issues
 • Survey Responses

Workbook Summary


,Sheet Name,Rows,Columns,Memory(MB)
0,Support Tickets,476,8,0.20
1,Jira Issues,554,22,0.75
2,Survey Responses,3019,73,11.94



Sheet Validation


,Sheet,Exists
0,Support Tickets,True
1,Jira Issues,True
2,Survey Responses,True



Schema Sample


...


Duplicate Sheet Names : False

Mandatory Column Validation


,Sheet,Column,Exists
0,Support Tickets,Ticket Id,True
1,Jira Issues,Key,True
2,Survey Responses,Response ID,True



Workbook Statistics


,Sheet,Rows,Columns,Missing Cells,Duplicate Rows,Numeric Columns,Categorical Columns,Date Columns
0,Support Tickets,476,8,110,0,1,7,0
1,Jira Issues,554,22,439,0,14,8,0
2,Survey Responses,3019,73,41791,0,6,67,0



Overall Validation Result : PASS
No validation issues detected.

Saving Reports...

Reports Generated Successfully!

Generated Files
output/
 ├── Validation_Report.xlsx
 └── Workbook_Summary.xlsx

Notebook 01 Completed Successfully
